# Analyse Exploratoire des Données — Marketing ROI

**Projet M1 Data Engineering & AI — EFREI 2025-26**  
**RNCP40875 Bloc 2 — Compétence C3.1 : Analyse des données**

Ce notebook présente l'EDA complète du dataset *Dummy Advertising & Sales Data* (Kaggle).  
Il s'appuie sur les fonctions de `src/data_preparation.py` pour garantir la cohérence avec le pipeline d'entraînement.

## 0. Configuration

In [ ]:
import os
import sys

# Ajouter la racine du projet dans le path pour importer src/
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

RESULTS_DIR = '../results'
DATA_PATH   = '../data/marketing_and_sales.csv'
print('Setup OK')

## 1. Chargement et ingénierie des features

`load_data()` :
- renomme `Social Media` → `Social_Media` (compatibilité sklearn)
- impute les NaN numériques par la médiane, `Influencer` par le mode

`engineer_features()` crée les variables dérivées :
- `Total_Budget = TV + Radio + Social_Media`
- `ROI = Sales / Total_Budget`
- Parts budgétaires : `TV_Share`, `Radio_Share`, `Social_Share`
- `Performance` : classification en terciles Low / Medium / High

In [ ]:
from src.data_preparation import (
    load_data, engineer_features,
    NUMERIC_FEATURES, TARGET_REG, TARGET_CLF,
    detect_outliers_iqr, check_quasi_constant,
    check_multicollinearity, find_saturation_points,
)

df_raw = load_data(DATA_PATH)
df     = engineer_features(df_raw)

print(f'Dimensions : {df.shape[0]} campagnes × {df.shape[1]} variables')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(3)

## 2. Valeurs manquantes

In [ ]:
missing = df.isnull().sum()
print('Valeurs manquantes après nettoyage :')
print(missing[missing > 0] if missing.any() else '  Aucune ✅')

## 3. Distributions des variables numériques

In [ ]:
cols_plot = NUMERIC_FEATURES + [TARGET_REG, 'Total_Budget', 'ROI']
colors = ['#e74c3c', '#2ecc71', '#9b59b6', '#3498db', '#f39c12', '#1abc9c']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col, color in zip(axes.flatten(), cols_plot, colors):
    ax.hist(df[col].dropna(), bins=25, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_xlabel('Valeur')
    ax.set_ylabel('Fréquence')

plt.suptitle('Distributions des variables numériques', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Distribution de la variable catégorielle Influencer

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['Influencer'].value_counts().plot(kind='bar', ax=axes[0], color='#95a5a6', edgecolor='white')
axes[0].set_title("Répartition des types d'influenceurs")
axes[0].set_ylabel('Nombre de campagnes')
axes[0].tick_params(axis='x', rotation=0)

df.boxplot(column=TARGET_REG, by='Influencer', ax=axes[1],
           color=dict(boxes='#3498db', whiskers='#3498db', medians='#e74c3c', caps='#3498db'))
axes[1].set_title('Ventes par type Influencer')
axes[1].set_ylabel('Sales (M€)')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 5. Distribution de la variable cible classification (Performance)

La variable `Performance` est construite par terciles sur `Sales` → classes équilibrées.

In [ ]:
perf_counts = df[TARGET_CLF].value_counts()
print('Distribution Performance :')
print(perf_counts)
print(f'\nÉquilibre : {perf_counts.min()/perf_counts.max():.2%} (1.0 = parfait)')

colors_perf = {'Low': '#e74c3c', 'Medium': '#f39c12', 'High': '#2ecc71'}
perf_counts.plot(kind='bar', color=[colors_perf[c] for c in perf_counts.index], edgecolor='white')
plt.title('Distribution des classes de performance')
plt.xlabel('Classe')
plt.ylabel('Nombre de campagnes')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Matrice de corrélation

**Observation clé** : TV est fortement corrélée aux ventes (r ≈ 0.9).  
Radio et Social_Media ont un impact modéré.  
Les budgets sont quasi-indépendants entre eux (faible multicolinéarité).

In [ ]:
corr_cols = NUMERIC_FEATURES + [TARGET_REG, 'Total_Budget', 'ROI']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax, square=True, linewidths=0.5)
ax.set_title('Matrice de corrélation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nCorrelations with Sales (sorted) :')
print(corr['Sales'].drop('Sales').sort_values(ascending=False).round(3))

## 7. Scatter Budgets → Ventes par canal

In [ ]:
influencer_colors = {'Mega': '#e74c3c', 'Macro': '#3498db', 'Micro': '#2ecc71', 'Nano': '#f39c12'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, channel in zip(axes, NUMERIC_FEATURES):
    for inf_type, color in influencer_colors.items():
        mask = df['Influencer'] == inf_type
        ax.scatter(df.loc[mask, channel], df.loc[mask, TARGET_REG],
                   color=color, alpha=0.6, label=inf_type, s=35)
    # Ligne de tendance
    z = np.polyfit(df[channel], df[TARGET_REG], 1)
    p = np.poly1d(z)
    xs = np.linspace(df[channel].min(), df[channel].max(), 100)
    ax.plot(xs, p(xs), 'k--', linewidth=1.5, alpha=0.6)
    ax.set_xlabel(f'Budget {channel} (M€)')
    ax.set_ylabel('Ventes (M€)')
    ax.set_title(f'{channel} → Sales')
    ax.legend(title='Influencer', fontsize=7)

plt.suptitle('Impact des budgets sur les ventes par type d\'influenceur', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. A10 — Détection des outliers (méthode IQR)

In [ ]:
outlier_df = detect_outliers_iqr(df)
print('Analyse des outliers par variable (méthode IQR, seuil 1.5×) :')
display(outlier_df)

n_total = outlier_df['N outliers'].sum()
if n_total == 0:
    print('\n✅ Aucun outlier — dataset synthétique bien contrôlé.')
else:
    print(f'\n⚠️  {n_total} outliers détectés. Impact à évaluer sur les modèles linéaires.')

## 9. A12 — Variables quasi-constantes

In [ ]:
qc_df = check_quasi_constant(df, threshold=0.95)
print('Variables quasi-constantes (seuil 95%) :')
display(qc_df)

quasi = qc_df[qc_df['Quasi-constante']]
if quasi.empty:
    print('\n✅ Toutes les features sont informatives.')
else:
    print(f'\n⚠️  Variables à considérer pour exclusion : {quasi["Variable"].tolist()}')

## 10. A23 — Analyse de multicolinéarité

In [ ]:
mc_df = check_multicollinearity(df, threshold=0.7)
print('Paires avec |r| > 0.7 :')
if mc_df.empty:
    print('  Aucune colinéarité problématique entre features ✅')
else:
    display(mc_df)
    print("\n  Note : Total_Budget est exclu des features du modèle (= TV+Radio+Social_Media par construction).")
    print("  La colinéarité TV→Sales est acceptable : TV est un prédicteur légitime, pas une feature redondante.")

## 11. A15 — Saturation budgétaire (rendement marginal décroissant)

In [ ]:
sat_df = find_saturation_points(df)
print('Points de saturation estimés :')
display(sat_df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, channel in zip(axes, NUMERIC_FEATURES):
    roi_col = df['Sales'] / df[channel].replace(0, np.nan)
    df_tmp = pd.DataFrame({channel: df[channel], 'roi': roi_col}).dropna()
    try:
        df_tmp['bin'] = pd.qcut(df_tmp[channel], q=10, duplicates='drop')
    except ValueError:
        continue
    grp = df_tmp.groupby('bin', observed=True).agg(
        budget_mean=(channel, 'mean'), roi_mean=('roi', 'mean')
    ).reset_index()

    ax.plot(grp['budget_mean'], grp['roi_mean'], 'o-', color='#3498db', linewidth=2)
    ax.fill_between(grp['budget_mean'], grp['roi_mean'], alpha=0.1, color='#3498db')
    ax.set_xlabel(f'Budget {channel} (M€)')
    ax.set_ylabel('ROI (Sales / Budget canal)')
    ax.set_title(f'Rendement marginal — {channel}')

    row = sat_df[sat_df['Canal'] == channel]
    if not row.empty:
        val = row.iloc[0]['Budget saturation estimé (M€)']
        if val not in ('Non détecté', 'Non calculable'):
            sb = float(val)
            ax.axvline(sb, color='crimson', linestyle='--', linewidth=1.5,
                       label=f'Saturation ≈ {sb:.0f} M€')
            ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

plt.suptitle('Rendement marginal décroissant par canal', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 12. Analyse de la variable cible vs canaux

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
perf_colors = {'Low': '#e74c3c', 'Medium': '#f39c12', 'High': '#2ecc71'}

for ax, channel in zip(axes, NUMERIC_FEATURES):
    for perf, color in perf_colors.items():
        mask = df['Performance'] == perf
        ax.scatter(df.loc[mask, channel], df.loc[mask, 'Sales'],
                   color=color, alpha=0.5, label=perf, s=25)
    ax.set_xlabel(f'Budget {channel} (M€)')
    ax.set_ylabel('Ventes (M€)')
    ax.set_title(f'{channel} vs Sales par Performance')
    ax.legend(title='Performance', fontsize=8)

plt.suptitle('Séparation des classes de performance selon les budgets', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 13. Synthèse EDA

| Observation | Implication pour la modélisation |
|-------------|----------------------------------|
| TV fortement corrélée aux ventes (r ≈ 0.9) | Feature dominante — à conserver en priorité |
| Social Media meilleur ROI marginal à faible budget | Non-linéarité → favorise RF / GB sur LR |
| Aucun outlier, aucune quasi-constante | Pas de nettoyage agressif nécessaire |
| Classes Performance équilibrées (terciles) | Pas de déséquilibre sévère — Accuracy fiable |
| Saturation TV ≈ 200–250 M€ | Rendement marginal décroissant — validation du modèle non-linéaire |
| Multicolinéarité Total_Budget / TV | Total_Budget exclu des features (construit = TV+R+S) |

**Conclusion** : le dataset est propre, synthétique, sans bruit excessif.  
Les R² > 0.99 observés lors de la modélisation s'expliquent par ce caractère synthétique.  
Sur des données réelles, les R² attendus seraient de l'ordre de 0.70–0.90.